# Social Connections Needs by Kansas County

This notebook explores social connection needs across Kansas counties. It joins processed patient information with department records and social-determinant responses, summarizes patient counts by provider, ZIP code, and county, and plots county-level counts for social connections responses.

**Inputs:** `data/private/processed/foot.csv`, `data/private/raw/departments.csv`, and `data/private/processed/sd_clean.csv`.

**Geographic boundaries:** `data/geospatial/kansas_tracts.geojson` and `data/geospatial/kansas_counties.geojson` (used by the optional map block).


In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import folium
import webbrowser

# Find the Bigfoot repository from the current working directory.
current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        folder for folder in (current_dir, *current_dir.parents)
        if (folder / "data").is_dir() and (folder / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the Bigfoot project root. Open Jupyter inside the Bigfoot repository."
    )

RAW_DIR = PROJECT_ROOT / "data" / "private" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "private" / "processed"
GEO_DIR = PROJECT_ROOT / "data" / "geospatial"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
clean_df = pd.read_csv(PROCESSED_DIR / "foot.csv")
clean_df.info()


In [ ]:
map_df=clean_df.copy()
map_df['access_score']=0
map_df['GEOID']=map_df["GEOID"].astype(str)
map_df["county"] = map_df["GEOID"].str[:5]

map_df.head()
map_df.shape

In [ ]:
# Load existing Kansas Census GeoJSON files rather than regenerating
# them from machine-specific shapefile paths.
gdf = gpd.read_file(GEO_DIR / "kansas_tracts.geojson")
gdf_kansas = gpd.read_file(GEO_DIR / "kansas_counties.geojson")


In [ ]:

"""
# --- Create map ---
m2 = folium.Map(location=[38.5, -98], zoom_start=7, tiles='CartoDB positron')

# --- Census Tracts Layer ---
folium.Choropleth(
    geo_data=str(GEO_DIR / "kansas_tracts.geojson"),
    name="Census Tracts",  
    data=map_df,
    columns=["GEOID", "access_score"],
    key_on="feature.properties.GEOID",
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score"
).add_to(m2)

# --- Counties Layer ---
county_choropleth=folium.Choropleth(
    geo_data=str(GEO_DIR / "kansas_counties.geojson"),
    name="Counties",  
    data=map_df,
    columns=["GEOID", "access_score"],
    key_on="feature.properties.GEOID",
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score",
    show=False  
).add_to(m2)

# --- County popup (click to see name) ---
folium.GeoJsonPopup(
        fields=["NAME"],         
        aliases=["County:"],
        localize=True
    ).add_to(county_choropleth.geojson)


# --- Layer toggle ---
folium.LayerControl().add_to(m2)

# --- Save and open ---
m2.save(str(OUTPUT_DIR / "social_county_map.html"))
webbrowser.open((OUTPUT_DIR / "social_county_map.html").resolve().as_uri())"""

In [ ]:
dept_df = pd.read_csv(RAW_DIR / "departments.csv")


In [ ]:
map_df.head()

In [ ]:
dept_df=dept_df.dropna(subset=['CensusTract'])
dept_df.head()

In [ ]:
dept_df['CensusTract']=dept_df['CensusTract'].astype(float).astype('Int64').astype(str)
dept_df.head()

In [ ]:
dept_df.dtypes

#pd.merge(map_df,dept_df,left_on='GEOID',right_on='CensusTract')

In [ ]:
map_df['GEOID'].dtypes

In [ ]:
len('201770031002')

In [ ]:
len('20177003603')

In [ ]:
map_df.head()
map_df['GEOID'] = map_df['GEOID'].astype(str)

# create tract (first 11 digits)
map_df['tract'] = map_df['GEOID'].str[:11]

# create block (last digit)
map_df['block'] = map_df['GEOID'].str[-1]
map_df.head()

In [ ]:
merged_df=pd.merge(map_df,dept_df,left_on='tract',right_on='CensusTract')

In [ ]:
merged_df.shape

In [ ]:
merged_df.head()

In [ ]:
merged_df.columns

In [ ]:
# Count patients per provider
patient_counts = merged_df.groupby('ProviderDurableKey')['PatientDurableKey'].count()

print(patient_counts)

In [ ]:
# Count unique patients per provider
patient_counts = merged_df.groupby('ProviderDurableKey')['PatientDurableKey'].nunique().reset_index()
patient_counts.columns = ['Provider', 'Patient_Count']
print(patient_counts)

In [ ]:
total_patients = merged_df['PatientDurableKey'].nunique()
total_providers = merged_df['ProviderDurableKey'].nunique()

ratio = total_patients / total_providers
print(f"Average Patients per Provider: {ratio:.2f}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.barplot(x='Patient_Count', y='Provider', data=patient_counts)
plt.title('Patients Assigned per Provider')
plt.show()


In [ ]:
merged_df.info()

In [ ]:
# Count unique patients per zipcode
patient_zip = merged_df.groupby('PostalCode')['PatientDurableKey'].nunique().reset_index()
patient_zip.columns = ['PostalCode', 'Patient_Count']
print(patient_zip)

In [ ]:
total_patient_zip = merged_df['PatientDurableKey'].nunique()
total_postal_code = merged_df['PostalCode'].nunique()

ratio = total_patient_zip / total_postal_code
print(f"Average Patients per Postal Code: {ratio:.2f}")

In [ ]:
sns.barplot(x='PostalCode', y='Patient_Count', data=patient_zip)
plt.title('Patients Assigned per Postal Code')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Combine the two columns into one new 'Final_County' column
# This takes 'County' first; if it's missing, it grabs 'county'
merged_df['Final_County'] = merged_df['County'].combine_first(merged_df['county'])

# Optional: Drop the old messy columns to clean up dataframe
merged_df = merged_df.drop(columns=['county', 'County'])

In [ ]:
# Count unique patients per county
patient_county = merged_df.groupby('Final_County')['PatientDurableKey'].nunique().reset_index()
patient_county.columns = ['Final_County', 'Patient_Count']
print(patient_county)

In [ ]:
total_patient_county = merged_df['PatientDurableKey'].nunique()
total_postal_county = merged_df['Final_County'].nunique()

ratio = total_patient_county / total_postal_county
print(f"Average Patients per County: {ratio:.2f}")

In [ ]:
sns.barplot(x='Final_County', y='Patient_Count', data=patient_county)
plt.title('Patients Assigned per County')
plt.xticks(rotation=90)
plt.show()

In [ ]:
social = pd.read_csv(PROCESSED_DIR / "sd_clean.csv")
social.info()

In [ ]:
social.drop(social.columns[[0]], axis=1, inplace=True)
social.info()

In [ ]:
# Filter the dataset to keep only rows where the domain column is 'Yes'
social_Filter = social[social['AnswerText'] == 'No']

social_Filter.info()

In [ ]:
social_Filter = social_Filter[(social_Filter['Domain'] == 'social connections') & (social_Filter['AnswerText'] == 'No')]

In [ ]:
S_merged = pd.merge(merged_df, social_Filter, on=['PatientDurableKey'], how='inner')
S_merged.info()

In [ ]:
print(S_merged['Domain'])

In [ ]:
# 1. Filter for the specific domain and the 'yes' response
S_merged = S_merged[(S_merged['Domain'] == 'social connections') & (S_merged['AnswerText'] == 'No')]

# 2. Group by county and count unique patient IDs
county_counts_S = S_merged.groupby('Final_County')['PatientDurableKey'].nunique().reset_index()

# 3. Rename columns for clarity
county_counts_S.columns = ['County', 'Unique_Patient_Count']

print(county_counts_S)

In [ ]:
county_counts_S.plot(kind='bar', x='County', y='Unique_Patient_Count', figsize=(10, 6), color='skyblue', edgecolor='navy')
plt.title('Unique Patients with Unmet Social Connections Needs by County')
plt.xlabel('County')
plt.ylabel('Number of Unique Patients')
plt.xticks(rotation=45) 
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()